# Churn Drivers Analysis

## Objectives
- Identify the most significant predictors of churn.
- Use statistical tests (Chi-Square, T-Test) to validate relationships.
- Rank features by importance using Mutual Information.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import chi2_contingency, ttest_ind
from sklearn.feature_selection import mutual_info_classif
from sklearn.preprocessing import LabelEncoder

pd.set_option('display.max_columns', None)
plt.style.use('seaborn-v0_8-whitegrid')

In [ ]:
# Load Data
df = pd.read_csv('../data/processed/final_analytical_dataset.csv')

## 1. Statistical Tests
### Categorical Variables (Chi-Square Test)

In [ ]:
cat_cols = ['Contract', 'PaymentMethod', 'InternetService', 'TechSupport', 'OnlineSecurity', 'Gender']
# Note: Gender might be encoded as 0/1, check dtype
if df['gender'].dtype != 'O':
    cat_cols = [c for c in cat_cols if c != 'Gender']

results = []
for col in cat_cols:
    if col in df.columns:
        contingency_table = pd.crosstab(df[col], df['Churn'])
        chi2, p, dof, expected = chi2_contingency(contingency_table)
        results.append({'Feature': col, 'Chi2': chi2, 'P-Value': p})

stat_df = pd.DataFrame(results).sort_values(by='Chi2', ascending=False)
stat_df

### Numerical Variables (T-Test)

In [ ]:
num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges', 'Weekly_Engagement_Score', 'Historical_CLV']

results_num = []
for col in num_cols:
    churn_vals = df[df['Churn'] == 1][col]
    non_churn_vals = df[df['Churn'] == 0][col]
    t_stat, p_val = ttest_ind(churn_vals, non_churn_vals, nan_policy='omit')
    results_num.append({'Feature': col, 'T-Stat': t_stat, 'P-Value': p_val})

stat_num_df = pd.DataFrame(results_num).sort_values(by='T-Stat', key=abs, ascending=False)
stat_num_df

## 2. Mutual Information Score

In [ ]:
# Prepare data for MI
X = df.drop(['customerID', 'Churn', 'Tenure_Cohort', 'Last_Activity_Date'], axis=1, errors='ignore')
y = df['Churn']

# Encode categorical for MI
for col in X.select_dtypes(include='object').columns:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].astype(str))

# Fill NaNs
X = X.fillna(0)

mi_scores = mutual_info_classif(X, y, random_state=42)
mi_df = pd.DataFrame({'Feature': X.columns, 'MI_Score': mi_scores}).sort_values(by='MI_Score', ascending=False)

plt.figure(figsize=(12, 8))
sns.barplot(x='MI_Score', y='Feature', data=mi_df.head(20), palette='viridis')
plt.title('Top 20 Features by Mutual Information')
plt.show()